In [1]:
import requests
import pandas as pd 
FRED_API = "97d6b8338b5fa5d6a030763ca042edc4"
sids = {"DCOILWTICO","CPIUFDSL", "WPU08", "UNRATE", "GDPC1",}
a11 = {} #getting a dictionary where keys are {"UNRATE" : }
for sid in sids:
    #to get a response for each sid, just grouping the process together 
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={sid}&api_key={FRED_API}&file_type=json"
    print(url)
    res=requests.get(url)
    res_json = res.json()

#to extract values from the response in 2023
    t = []
    for o in res_json["observations"]:
        if "2023" in o["date"]:
            t.append(o["value"])
   #t which is a list of the values in 2023
   #and we know sid is each series_id
    a11[sid] = t

https://api.stlouisfed.org/fred/series/observations?series_id=WPU08&api_key=97d6b8338b5fa5d6a030763ca042edc4&file_type=json
https://api.stlouisfed.org/fred/series/observations?series_id=UNRATE&api_key=97d6b8338b5fa5d6a030763ca042edc4&file_type=json
https://api.stlouisfed.org/fred/series/observations?series_id=GDPC1&api_key=97d6b8338b5fa5d6a030763ca042edc4&file_type=json
https://api.stlouisfed.org/fred/series/observations?series_id=DCOILWTICO&api_key=97d6b8338b5fa5d6a030763ca042edc4&file_type=json
https://api.stlouisfed.org/fred/series/observations?series_id=CPIUFDSL&api_key=97d6b8338b5fa5d6a030763ca042edc4&file_type=json


## US EURO

In [2]:
# Initialize an empty dictionary to store the data for each series
data = {}

# Define the series IDs we are interested in
series_ids = ["CPIUFDSL", "WPU08", "UNRATE"]

# Fetch data for each series
for sid in series_ids:
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={sid}&api_key={FRED_API}&file_type=json"
    res = requests.get(url)
    res_json = res.json()
    
    # Extract date and value pairs
    dates = []
    values = []
    for o in res_json["observations"]:
        date = o["date"]
        value = o["value"]
        if date >= "2000-01-01":
            dates.append(date)
            values.append(value)
    
    # Store the data in the dictionary
    data[sid] = pd.Series(values, index=dates)

# Create a DataFrame from the dictionary
df = pd.DataFrame(data)

# Drop rows with NaN values to ensure all series have data for the same dates
df.dropna(inplace=True)

# Reset the index to have the date as a column
df.reset_index(inplace=True)
df.rename(columns={'index': 'date'}, inplace=True)

# Change the date format to yyyy-mm
df['date'] = pd.to_datetime(df['date']).dt.to_period('M').astype(str)

# Display the DataFrame
df

,date,CPIUFDSL,WPU08,UNRATE
0,2000-01,165.6,183.8,4.0
1,2000-02,166.2,184.0,4.1
2,2000-03,166.5,184.2,4.0
3,2000-04,166.7,183.0,3.8
4,2000-05,167.3,179.3,4.0
...,...,...,...,...
303,2025-04,337.424,302.378,4.2
304,2025-05,338.386,301.292,4.2
305,2025-06,339.498,300.209,4.1
306,2025-07,339.652,300.228,4.2


In [3]:
series_id = "DCOILWTICO"

# Fetch data for the series
url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={FRED_API}&file_type=json"
res = requests.get(url)
res_json = res.json()

# Extract date and value pairs
dates = []
values = []
for o in res_json["observations"]:
    date = o["date"]
    value = o["value"]
    if date >= "2000-01-01":
        dates.append(date)
        values.append(value)

# Create a DataFrame from the extracted data
dcoilwtico_df = pd.DataFrame({"date": dates, "DCOILWTICO": values})

# Convert the date column to datetime format and set the format to yyyy-mm
dcoilwtico_df["date"] = pd.to_datetime(dcoilwtico_df["date"]).dt.to_period('M').astype(str)

# Convert DCOILWTICO column to numeric, forcing errors to NaN
dcoilwtico_df["DCOILWTICO"] = pd.to_numeric(dcoilwtico_df["DCOILWTICO"], errors='coerce')

# Group by the date and calculate the mean for each month
monthly_avg_df = dcoilwtico_df.groupby("date").mean().reset_index()

# Filter out rows after 2024-10
monthly_avg_df = monthly_avg_df[monthly_avg_df["date"] <= "2024-10"]

# Display the DataFrame
monthly_avg_df

,date,DCOILWTICO
0,2000-01,27.259474
1,2000-02,29.366000
2,2000-03,29.841739
3,2000-04,25.722105
4,2000-05,28.788182
...,...,...
293,2024-06,79.767368
294,2024-07,81.800455
295,2024-08,76.683182
296,2024-09,70.236000


In [4]:
# Convert the 'date' column in df to datetime format and set the format to yyyy-mm
df["date"] = pd.to_datetime(df["date"]).dt.to_period('M').astype(str)

# Merge the two DataFrames on the 'date' column
merged_df = pd.merge(df, monthly_avg_df, on="date", how="inner")

# Display the merged DataFrame
merged_df

,date,CPIUFDSL,WPU08,UNRATE,DCOILWTICO
0,2000-01,165.6,183.8,4.0,27.259474
1,2000-02,166.2,184.0,4.1,29.366000
2,2000-03,166.5,184.2,4.0,29.841739
3,2000-04,166.7,183.0,3.8,25.722105
4,2000-05,167.3,179.3,4.0,28.788182
...,...,...,...,...,...
293,2024-06,329.651,296.089,4.1,79.767368
294,2024-07,330.142,294.402,4.2,81.800455
295,2024-08,330.644,295.416,4.2,76.683182
296,2024-09,331.716,296.369,4.1,70.236000


In [5]:
# Define the series ID for GDP
GDP_API = "GDPC1"

# Fetch data for the series
url2 = f"https://api.stlouisfed.org/fred/series/observations?series_id={GDP_API}&api_key={FRED_API}&file_type=json"
res2 = requests.get(url2)
res_json2 = res2.json()

# Extract date and value pairs starting from 1947
dates = []
values = []
for o in res_json2["observations"]:
    date = o["date"]
    value = o["value"]
    if date >= "1947-01-01":
        dates.append(date)
        values.append(value)

# Create a DataFrame from the extracted data
gdp_df = pd.DataFrame({
    "date": dates,
    "GDP": values
})

# Convert GDP values to numeric
gdp_df["GDP"] = pd.to_numeric(gdp_df["GDP"], errors='coerce')

# Calculate percent change
gdp_df["GDP_pct_change"] = gdp_df["GDP"].pct_change()*100

# Drop the first row
gdp_df = gdp_df.dropna().reset_index(drop=True)

# Drop the GDP column and rename GDP_pct_change to GDP%
gdp_df = gdp_df.drop(columns=["GDP"]).rename(columns={"GDP_pct_change": "GDP%"})

# Display the DataFrame
gdp_df

,date,GDP%
0,1947-04-01,-0.265224
1,1947-07-01,-0.204879
2,1947-10-01,1.565987
3,1948-01-01,1.506038
4,1948-04-01,1.652377
...,...,...
308,2024-04-01,0.885486
309,2024-07-01,0.824778
310,2024-10-01,0.459875
311,2025-01-01,-0.162516


In [6]:
# Filter out rows before the year 2000
gdp_df2000 = gdp_df[gdp_df["date"] >= "2000-01-01"].reset_index(drop=True)

# Display the filtered DataFrame
gdp_df2000

,date,GDP%
0,2000-01-01,0.362793
1,2000-04-01,1.821288
2,2000-07-01,0.101933
3,2000-10-01,0.597039
4,2001-01-01,-0.327799
...,...,...
97,2024-04-01,0.885486
98,2024-07-01,0.824778
99,2024-10-01,0.459875
100,2025-01-01,-0.162516


In [7]:
import pandas as pd

# Assuming gdp_df2000 is already defined and is a DataFrame

# Create a list to store the updated values
data = []

# Iterate over each row in gdp_df2000
for index, row in gdp_df2000.iterrows():
    # Calculate the new GDP% value
    new_gdp_value = row["GDP%"] / 3
    
    # Create new rows with updated dates and GDP% values
    for i in range(3):
        new_date = pd.to_datetime(row["date"]) + pd.DateOffset(months=i)
        data.append({"date": new_date.strftime("%Y-%m"), "GDP%": new_gdp_value})

# Add the final row with the specified date and GDP%
data.append({"date": "2024-10", "GDP%": 1.06666})

# Create a new DataFrame from the list
gdp_df2 = pd.DataFrame(data, columns=["date", "GDP%"])

# Display the new DataFrame
gdp_df2

,date,GDP%
0,2000-01,0.120931
1,2000-02,0.120931
2,2000-03,0.120931
3,2000-04,0.607096
4,2000-05,0.607096
...,...,...
302,2025-03,-0.054172
303,2025-04,0.315333
304,2025-05,0.315333
305,2025-06,0.315333


In [8]:
# Merge gdp_df2 with merged_df on the 'date' column
Final_US_DATA = pd.merge(merged_df, gdp_df2, on="date", how="inner")
# Display the merged DataFrame
Final_US_DATA

,date,CPIUFDSL,WPU08,UNRATE,DCOILWTICO,GDP%
0,2000-01,165.6,183.8,4.0,27.259474,0.120931
1,2000-02,166.2,184.0,4.1,29.366000,0.120931
2,2000-03,166.5,184.2,4.0,29.841739,0.120931
3,2000-04,166.7,183.0,3.8,25.722105,0.607096
4,2000-05,167.3,179.3,4.0,28.788182,0.607096
...,...,...,...,...,...,...
294,2024-07,330.142,294.402,4.2,81.800455,0.274926
295,2024-08,330.644,295.416,4.2,76.683182,0.274926
296,2024-09,331.716,296.369,4.1,70.236000,0.274926
297,2024-10,332.418,297.748,4.1,71.985000,0.153292


In [9]:
# Import necessary libraries
import pandas as pd

# Assuming Final_US_DATA is already defined and is a DataFrame

# Convert 'date' column to datetime format for better handling
Final_US_DATA['date'] = pd.to_datetime(Final_US_DATA['date'])

# Sort the DataFrame by date
Final_US_Data = Final_US_DATA.sort_values(by='date').reset_index(drop=True)

# Rename columns for better readability and remove spaces
Final_US_Data = Final_US_Data.rename(columns={
    "CPIUFDSL": "Food_CPI",
    "WPU08": "Lumber_PPI",
    "UNRATE": "Unemployment_Rate",
    "DCOILWTICO": "Crude_Oil_Prices",
    "GDP%": "GDP_Growth_Rate",
    "date": "Date"
})

# Ensure the 'Date' column is in yyyy-mm format
Final_US_Data['Date'] = Final_US_Data['Date'].dt.strftime('%Y-%m')

# Reorder columns for better presentation
Final_US_Data = Final_US_Data[["Date", "Food_CPI", "Lumber_PPI", "Unemployment_Rate", "Crude_Oil_Prices", "GDP_Growth_Rate"]]

# Display the improved DataFrame
Final_US_Data

,Date,Food_CPI,Lumber_PPI,Unemployment_Rate,Crude_Oil_Prices,GDP_Growth_Rate
0,2000-01,165.6,183.8,4.0,27.259474,0.120931
1,2000-02,166.2,184.0,4.1,29.366000,0.120931
2,2000-03,166.5,184.2,4.0,29.841739,0.120931
3,2000-04,166.7,183.0,3.8,25.722105,0.607096
4,2000-05,167.3,179.3,4.0,28.788182,0.607096
...,...,...,...,...,...,...
294,2024-07,330.142,294.402,4.2,81.800455,0.274926
295,2024-08,330.644,295.416,4.2,76.683182,0.274926
296,2024-09,331.716,296.369,4.1,70.236000,0.274926
297,2024-10,332.418,297.748,4.1,71.985000,0.153292


In [10]:
# Display the data types of each column in Final_US_Data
Final_US_Data.dtypes

Date                  object
Food_CPI              object
Lumber_PPI            object
Unemployment_Rate     object
Crude_Oil_Prices     float64
GDP_Growth_Rate      float64
dtype: object

In [11]:
# Convert all columns except 'Date' to float
for col in Final_US_Data.columns:
    if col != 'Date':
        Final_US_Data[col] = Final_US_Data[col].astype(float)

# Convert 'Date' column back to datetime format
Final_US_Data['Date'] = pd.to_datetime(Final_US_Data['Date'], format='%Y-%m')

# Display the data types of each column in Final_US_Data
Final_US_Data.dtypes

Date                 datetime64[ns]
Food_CPI                    float64
Lumber_PPI                  float64
Unemployment_Rate           float64
Crude_Oil_Prices            float64
GDP_Growth_Rate             float64
dtype: object

In [12]:
Final_US_Data# Ensure the 'Date' column is in datetime format
Final_US_Data['Date'] = pd.to_datetime(Final_US_Data['Date'])

# Format the 'Date' column to 'yyyy-mm'
Final_US_Data['Date'] = Final_US_Data['Date'].dt.strftime('%Y-%m')

Final_US_Data

,Date,Food_CPI,Lumber_PPI,Unemployment_Rate,Crude_Oil_Prices,GDP_Growth_Rate
0,2000-01,165.600,183.800,4.0,27.259474,0.120931
1,2000-02,166.200,184.000,4.1,29.366000,0.120931
2,2000-03,166.500,184.200,4.0,29.841739,0.120931
3,2000-04,166.700,183.000,3.8,25.722105,0.607096
4,2000-05,167.300,179.300,4.0,28.788182,0.607096
...,...,...,...,...,...,...
294,2024-07,330.142,294.402,4.2,81.800455,0.274926
295,2024-08,330.644,295.416,4.2,76.683182,0.274926
296,2024-09,331.716,296.369,4.1,70.236000,0.274926
297,2024-10,332.418,297.748,4.1,71.985000,0.153292


In [13]:
# Set the base year for scaling
base_year = '2000-01'

# Ensure the 'Date' column is in datetime format
Final_US_Data['Date'] = pd.to_datetime(Final_US_Data['Date'], format='%Y-%m')

# Get the base values for Food_CPI and Lumber_PPI
base_food_cpi = Final_US_Data.loc[Final_US_Data['Date'] == base_year, 'Food_CPI'].values[0]
base_lumber_ppi = Final_US_Data.loc[Final_US_Data['Date'] == base_year, 'Lumber_PPI'].values[0]

# Scale the Food_CPI and Lumber_PPI columns based on the base year
Final_US_Data['Food_CPI'] = Final_US_Data['Food_CPI'] / base_food_cpi * 100
Final_US_Data['Lumber_PPI'] = Final_US_Data['Lumber_PPI'] / base_lumber_ppi * 100

# Format the 'Date' column back to 'yyyy-mm'
Final_US_Data['Date'] = Final_US_Data['Date'].dt.strftime('%Y-%m')

Final_US_Data

,Date,Food_CPI,Lumber_PPI,Unemployment_Rate,Crude_Oil_Prices,GDP_Growth_Rate
0,2000-01,100.000000,100.000000,4.0,27.259474,0.120931
1,2000-02,100.362319,100.108814,4.1,29.366000,0.120931
2,2000-03,100.543478,100.217628,4.0,29.841739,0.120931
3,2000-04,100.664251,99.564744,3.8,25.722105,0.607096
4,2000-05,101.026570,97.551687,4.0,28.788182,0.607096
...,...,...,...,...,...,...
294,2024-07,199.361111,160.175190,4.2,81.800455,0.274926
295,2024-08,199.664251,160.726877,4.2,76.683182,0.274926
296,2024-09,200.311594,161.245375,4.1,70.236000,0.274926
297,2024-10,200.735507,161.995647,4.1,71.985000,0.153292


In [14]:
# Save Final_US_DATA to a CSV file
Final_US_Data.to_csv("Final_US_Data.csv", index=False)

### END OF US DATA

In [15]:
FRED_API3 = "97d6b8338b5fa5d6a030763ca042edc4"
sids = {"DCOILBRENTEU",}
a11 = {} #getting a dictionary where keys are {"UNRATE" : }
for sid in sids:
    #to get a response for each sid, just grouping the process together 
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={sid}&api_key={FRED_API3}&file_type=json"
    print(url)
    res=requests.get(url)
    res_json = res.json()
#to extract values from the response in 2023
    t = []
    for o in res_json["observations"]:
        if "2023" in o["date"]:
            t.append(o["value"])
   #t which is a list of the values in 2023
   #and we know sid is each series_id
    a11[sid] = t 

https://api.stlouisfed.org/fred/series/observations?series_id=DCOILBRENTEU&api_key=97d6b8338b5fa5d6a030763ca042edc4&file_type=json


In [16]:
# Define the series ID for DCOILBRENTEU
series_id = "DCOILBRENTEU"

# Fetch data for the series
url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={FRED_API3}&file_type=json"
res = requests.get(url)
res_json = res.json()

# Extract date and value pairs
dates = []
values = []
for o in res_json["observations"]:
    date = o["date"]
    value = o["value"]
    if date >= "2000-01-01":
        dates.append(date)
        values.append(value)

# Create a DataFrame from the extracted data
euro_oil_df = pd.DataFrame({"date": dates, "EURO_OIL": values})

# Convert the date column to datetime format and set the format to yyyy-mm
euro_oil_df["date"] = pd.to_datetime(euro_oil_df["date"]).dt.to_period('M').astype(str)

# Convert EURO_OIL column to numeric, forcing errors to NaN
euro_oil_df["EURO_OIL"] = pd.to_numeric(euro_oil_df["EURO_OIL"], errors='coerce')

# Group by the date and calculate the mean for each month
monthly_avg_df = euro_oil_df.groupby("date").mean().reset_index()

# Filter out rows after 2024-10
monthly_avg_df = monthly_avg_df[monthly_avg_df["date"] <= "2024-10"]

# Display the DataFrame
monthly_avg_df

,date,EURO_OIL
0,2000-01,25.511000
1,2000-02,27.775714
2,2000-03,27.486087
3,2000-04,22.764444
4,2000-05,27.737619
...,...,...
293,2024-06,82.246000
294,2024-07,85.153043
295,2024-08,80.355238
296,2024-09,74.016667


In [17]:
# Merge the monthly_avg_df with Final_US_Data on the 'date' column
Final_US_EURO_Data = Final_US_Data.merge(monthly_avg_df, left_on='Date', right_on='date', how='left')

# Drop the 'date' column from the merged DataFrame
Final_US_EURO_Data.drop(columns=['date'], inplace=True)

# Display the merged DataFrame
Final_US_EURO_Data

,Date,Food_CPI,Lumber_PPI,Unemployment_Rate,Crude_Oil_Prices,GDP_Growth_Rate,EURO_OIL
0,2000-01,100.000000,100.000000,4.0,27.259474,0.120931,25.511000
1,2000-02,100.362319,100.108814,4.1,29.366000,0.120931,27.775714
2,2000-03,100.543478,100.217628,4.0,29.841739,0.120931,27.486087
3,2000-04,100.664251,99.564744,3.8,25.722105,0.607096,22.764444
4,2000-05,101.026570,97.551687,4.0,28.788182,0.607096,27.737619
...,...,...,...,...,...,...,...
294,2024-07,199.361111,160.175190,4.2,81.800455,0.274926,85.153043
295,2024-08,199.664251,160.726877,4.2,76.683182,0.274926,80.355238
296,2024-09,200.311594,161.245375,4.1,70.236000,0.274926,74.016667
297,2024-10,200.735507,161.995647,4.1,71.985000,0.153292,75.632609


In [18]:
# Save the Final_US_EURO_Data DataFrame to a CSV file
Final_US_EURO_Data.to_csv('Final_US_EURO_Data.csv', index=False)

## Adding more data on

In [41]:
# If needed (uncomment the next line in a terminal/cell once):
# %pip install fredapi pandas numpy openpyxl python-dateutil

import os
from datetime import datetime
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
from fredapi import Fred

In [42]:
fred = Fred(api_key="97d6b8338b5fa5d6a030763ca042edc4")
series_ids = {
    "MSPUS": "Median Sales Price of Houses Sold",      # may need to use "MSPUS" instead if this fails
    "CPIAUCSL": "Consumer Price Index for All Urban Consumers",
    "MEHOINUSA672N": "Median Household Income",
    "FEDMINNFRWG": "Federal Minimum Wage",
    "DGS10": "10-Year Treasury Constant Maturity Rate",
    "DGS2": "2-Year Treasury Constant Maturity Rate",
    "PCEPI": "Personal Consumption Expenditures Price Index"
}

# Fetch and combine all series
fred_df = pd.DataFrame()

for series_id, description in series_ids.items():
    try:
        data = fred.get_series(series_id)
        fred_df[series_id] = data
        print(f"Successfully retrieved: {series_id} ({description})")
    except Exception as e:
        print(f"Could not retrieve {series_id}: {e}")

# Make sure index is a proper datetime
fred_df.index = pd.to_datetime(fred_df.index)
fred_df = fred_df.resample('M').last()   # monthly frequency
fred_df.head()


Successfully retrieved: MSPUS (Median Sales Price of Houses Sold)
Successfully retrieved: CPIAUCSL (Consumer Price Index for All Urban Consumers)
Successfully retrieved: MEHOINUSA672N (Median Household Income)
Successfully retrieved: FEDMINNFRWG (Federal Minimum Wage)
Successfully retrieved: DGS10 (10-Year Treasury Constant Maturity Rate)
Successfully retrieved: DGS2 (2-Year Treasury Constant Maturity Rate)
Successfully retrieved: PCEPI (Personal Consumption Expenditures Price Index)


C:\Users\dpchr\AppData\Local\Temp\ipykernel_5876\2245635379.py:25: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  fred_df = fred_df.resample('M').last()   # monthly frequency


,MSPUS,CPIAUCSL,MEHOINUSA672N,FEDMINNFRWG,DGS10,DGS2,PCEPI
1963-01-31,17800.0,30.44,NaN,1.15,NaN,NaN,15.997
1963-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1963-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1963-04-30,18000.0,30.48,NaN,1.15,3.95,NaN,16.000
1963-05-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
start = pd.Timestamp("2000-01-01")
end   = pd.Timestamp("2024-10-31")  # month-end target
target_index = pd.date_range(start, end, freq="M")  # month-end index


C:\Users\dpchr\AppData\Local\Temp\ipykernel_5876\1495452116.py:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  target_index = pd.date_range(start, end, freq="M")  # month-end index


In [44]:
def to_monthly_complete(s: pd.Series, name: str) -> pd.Series:
    """
    Convert a FRED series to monthly values on target_index:
      - Daily (or higher): monthly MEAN
      - Quarterly: repeat quarter's value for its months
      - Annual / irregular stepwise: repeat year's value for its months
    Guarantees no NaNs on target_index.
    """
    s = s.dropna().sort_index()
    s.index = pd.to_datetime(s.index)
    s.name = name

    # Step 1: start with monthly mean (works for daily & already-monthly)
    sm = s.resample("M").mean()

    # Step 2: fill remaining gaps using quarterly average repeated within quarter
    sq = s.resample("Q").mean()               # quarterly mean if data supports it
    sq_monthly = sq.resample("M").ffill()     # repeat quarter value within the quarter
    sm = sm.reindex(sm.index.union(sq_monthly.index))
    sm = sm.combine_first(sq_monthly)

    # Step 3: fill remaining gaps using annual average repeated within year
    sa = s.resample("A-DEC").mean()           # annual mean (calendar year)
    sa_monthly = sa.resample("M").ffill()     # repeat across months of the year
    sm = sm.reindex(sm.index.union(sa_monthly.index))
    sm = sm.combine_first(sa_monthly)

    # Step 4: restrict to the target monthly range and clean edges
    sm = sm.reindex(target_index)
    sm = sm.ffill().bfill()                   # edge safety (rare)

    # Final sanity
    if sm.isna().any():
        raise ValueError(f"{name}: still contains NaNs after monthly conversion.")

    sm.name = name
    return sm


In [45]:
cols = []

# MSPUS (Median Sales Price of Houses Sold) -- quarterly → repeat within quarter
s = fred.get_series("MSPUS")
cols.append(to_monthly_complete(s, "MSPUS"))

# CPIAUCSL (CPI, monthly already)
s = fred.get_series("CPIAUCSL")
cols.append(to_monthly_complete(s, "CPIAUCSL"))

# MEHOINUSA672N (Median Household Income, annual) → repeat within year
s = fred.get_series("MEHOINUSA672N")
cols.append(to_monthly_complete(s, "MEHOINUSA672N"))

# FEDMINNFRWG (Federal Minimum Wage, stepwise/annual-ish) → repeat within year/months
s = fred.get_series("FEDMINNFRWG")
cols.append(to_monthly_complete(s, "FEDMINNFRWG"))

# DGS10 (10Y Treasury, daily) → monthly MEAN
s = fred.get_series("DGS10")
cols.append(to_monthly_complete(s, "DGS10"))

# DGS2 (2Y Treasury, daily) → monthly MEAN
s = fred.get_series("DGS2")
cols.append(to_monthly_complete(s, "DGS2"))

# PCEPI (PCE price index, monthly)
s = fred.get_series("PCEPI")
cols.append(to_monthly_complete(s, "PCEPI"))

# Federal Funds Rate (daily → monthly average)
s = fred.get_series("FEDFUNDS")
cols.append(to_monthly_complete(s, "FEDFUNDS"))

# M2 Money Supply (weekly → monthly average)
s = fred.get_series("M2SL")
cols.append(to_monthly_complete(s, "M2SL"))

# Industrial Production Index (monthly)
s = fred.get_series("INDPRO")
cols.append(to_monthly_complete(s, "INDPRO"))

# Unemployment Rate (monthly)
s = fred.get_series("UNRATE")
cols.append(to_monthly_complete(s, "UNRATE"))

# Retail Sales (monthly)
s = fred.get_series("RSAFS")
cols.append(to_monthly_complete(s, "RSAFS"))


fred_df_monthly = pd.concat(cols, axis=1)
fred_df_monthly.index.name = "DATE"

# Ensure exact date bounds and no missing values
fred_df_monthly = fred_df_monthly.loc[(fred_df_monthly.index >= start) & (fred_df_monthly.index <= end)]
assert fred_df_monthly.index.equals(target_index), "Index mismatch with target monthly range."
assert not fred_df_monthly.isna().any().any(), "There are unexpected NaNs."

fred_df_monthly.head(), fred_df_monthly.tail()


C:\Users\dpchr\AppData\Local\Temp\ipykernel_5876\1574035760.py:14: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  sm = s.resample("M").mean()
C:\Users\dpchr\AppData\Local\Temp\ipykernel_5876\1574035760.py:17: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  sq = s.resample("Q").mean()               # quarterly mean if data supports it
C:\Users\dpchr\AppData\Local\Temp\ipykernel_5876\1574035760.py:18: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  sq_monthly = sq.resample("M").ffill()     # repeat quarter value within the quarter
C:\Users\dpchr\AppData\Local\Temp\ipykernel_5876\1574035760.py:23: FutureWarning: 'A-DEC' is deprecated and will be removed in a future version, please use 'YE-DEC' instead.
  sa = s.resample("A-DEC").mean()           # annual mean (calendar year)
C:\Users\dpchr\AppData\Local\Temp\ipykernel_5876\15740357

(               MSPUS  CPIAUCSL  MEHOINUSA672N  FEDMINNFRWG     DGS10  \
 DATE                                                                   
 2000-01-31  165300.0     169.3        71790.0         5.15  6.661000   
 2000-02-29  165300.0     170.0        71980.0         5.15  6.519500   
 2000-03-31  165300.0     171.0        71790.0         5.15  6.256522   
 2000-04-30  163200.0     170.9        71790.0         5.15  5.990526   
 2000-05-31  165300.0     171.2        71790.0         5.15  6.440455   
 
                 DGS2   PCEPI  FEDFUNDS    M2SL   INDPRO  UNRATE     RSAFS  
 DATE                                                                       
 2000-01-31  6.440000  72.961      5.45  4667.6  91.4092     4.0  261545.0  
 2000-02-29  6.610500  73.191      5.73  4680.9  91.7245     4.1  265686.0  
 2000-03-31  6.528261  73.505      5.85  4711.7  92.0830     4.0  269019.0  
 2000-04-30  6.403684  73.444      6.02  4767.8  92.6659     3.8  264067.0  
 2000-05-31  6.809545  73

In [46]:
fred_df_monthly = fred_df_monthly.rename(columns={
    "MSPUS": "Median_Sales_Price_Houses_Sold",
    "CPIAUCSL": "Consumer_Price_Index_All_Urban_Consumers",
    "MEHOINUSA672N": "Median_Household_Income",
    "FEDMINNFRWG": "Federal_Minimum_Wage",
    "DGS10": "10_Year_Treasury_Rate",
    "DGS2": "2_Year_Treasury_Rate",
    "PCEPI": "Personal_Consumption_Expenditures_Price_Index",
    "FEDFUNDS": "Federal_Funds_Rate",
    "M2SL": "Money_Supply_M2",
    "INDPRO": "Industrial_Production_Index",
    "UNRATE": "Unemployment_Rate",
    "RSAFS": "Retail_Sales"
})

fred_df_monthly.index = fred_df_monthly.index.to_period('M').astype(str)
fred_df_monthly.index.name = "DATE"

# Forward/backward fill just to be safe (though your function already ensures no NaNs)
fred_df_monthly = fred_df_monthly.ffill().bfill()



In [47]:
fred_df_monthly

,Median_Sales_Price_Houses_Sold,Consumer_Price_Index_All_Urban_Consumers,Median_Household_Income,Federal_Minimum_Wage,10_Year_Treasury_Rate,2_Year_Treasury_Rate,Personal_Consumption_Expenditures_Price_Index,Federal_Funds_Rate,Money_Supply_M2,Industrial_Production_Index,Unemployment_Rate,Retail_Sales
DATE,,,,,,,,,,,,
2000-01,165300.0,169.300,71790.0,5.15,6.661000,6.440000,72.961,5.45,4667.6,91.4092,4.0,261545.0
2000-02,165300.0,170.000,71980.0,5.15,6.519500,6.610500,73.191,5.73,4680.9,91.7245,4.1,265686.0
2000-03,165300.0,171.000,71790.0,5.15,6.256522,6.528261,73.505,5.85,4711.7,92.0830,4.0,269019.0
2000-04,163200.0,170.900,71790.0,5.15,5.990526,6.403684,73.444,6.02,4767.8,92.6659,3.8,264067.0
2000-05,165300.0,171.200,71790.0,5.15,6.440455,6.809545,73.505,6.27,4755.7,92.9347,4.0,265992.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2024-06,414500.0,313.131,82690.0,7.25,4.305263,4.736842,123.539,5.33,21052.9,103.2534,4.1,692449.0
2024-07,415300.0,313.566,82690.0,7.25,4.248636,4.495909,123.736,5.33,21084.2,102.5192,4.2,698835.0
2024-08,414500.0,314.131,82690.0,7.25,3.870909,3.965455,123.889,5.33,21171.1,103.0196,4.2,697157.0


In [48]:
fred_df_monthly.to_csv("US_ECON_ADDITIONAL.csv", index=True)
print("Saved successfully as US_ECON_FULL.csv with DATE formatted as YYYY-MM.")


Saved successfully as US_ECON_FULL.csv with DATE formatted as YYYY-MM.


## Merging

In [50]:
import pandas as pd

# Load both datasets
final_df = pd.read_csv("Final_US_EURO_Data.csv")
additional_df = pd.read_csv("US_ECON_ADDITIONAL.csv")

# Standardize column names (strip spaces and uppercase everything)
final_df.columns = final_df.columns.str.strip().str.upper()
additional_df.columns = additional_df.columns.str.strip().str.upper()

# Now both have a column called "DATE"
final_df['DATE'] = final_df['DATE'].astype(str)
additional_df['DATE'] = additional_df['DATE'].astype(str)

# Merge on DATE
merged_df = pd.merge(final_df, additional_df, on="DATE", how="left")

# Check shape and missing values
print("Merged shape:", merged_df.shape)
print("Any NaNs?", merged_df.isna().any().any())

# Save the merged dataset
merged_df.to_csv("FINAL_US_EURO_FULL.csv", index=False)
print("Saved successfully as FINAL_US_EURO_FULL.csv")


Merged shape: (299, 19)
Any NaNs? False
Saved successfully as FINAL_US_EURO_FULL.csv


In [ ]:
merged_df